# RMSProp — Divide the gradient by a running average of its recent magnitude (2012)

_Papers · Foundations & Optimization_

**Keep a moving average of each weight's squared gradient and step by the gradient over its square root — so the step stays a healthy size instead of decaying to zero like AdaGrad's.**

---

This notebook is a practice scaffold for the **RMSProp — Divide the gradient by a running average of its recent magnitude (2012)** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np
import matplotlib.pyplot as plt

## Reference implementation — PyTorch

### Step 1 — Implement RMSProp from scratch

RMSProp (Tieleman & Hinton, 2012) keeps a per-weight moving average of the squared gradient — the *MeanSquare* `s_t = rho*s_{t-1} + (1-rho)*g^2` — and then steps by the gradient divided by its root mean square: `theta -= lr * g / (sqrt(s_t) + eps)`. Dividing by the recent gradient magnitude keeps every weight's step a healthy size. We build it as a small optimizer class with `step` and `zero_grad`, mirroring the PyTorch API.

In [ ]:
import torch

torch.manual_seed(0)

class MyRMSprop:
    """RMSProp from scratch — Tieleman & Hinton (2012), Coursera lecture 6e.
       s_t = rho*s_{t-1} + (1-rho)*g^2 ;  theta -= lr * g / (sqrt(s_t) + eps)."""
    def __init__(self, params, lr=1e-2, rho=0.99, eps=1e-8):
        self.params = list(params)
        self.lr, self.rho, self.eps = lr, rho, eps
        self.s = [torch.zeros_like(p) for p in self.params]   # MeanSquare (init 0)

    @torch.no_grad()
    def step(self):
        for i, p in enumerate(self.params):
            g = p.grad
            self.s[i] = self.rho * self.s[i] + (1 - self.rho) * g * g  # MeanSquare moving average
            p -= self.lr * g / (self.s[i].sqrt() + self.eps)           # divide grad by RMS (eps OUTSIDE sqrt)

    def zero_grad(self):
        for p in self.params:
            if p.grad is not None:
                p.grad.zero_()

### Step 2 — Check it against PyTorch's RMSprop

The oracle for correctness is `torch.optim.RMSprop`: starting both optimizers from an identical weight tensor and feeding identical gradients, our implementation must track PyTorch's to within floating-point tolerance over several steps. One naming gotcha — PyTorch calls the decay rate `alpha` (default 0.99) where we call it `rho`.

In [ ]:
# THE ORACLE: MyRMSprop must equal torch.optim.RMSprop over several steps.
# NOTE: PyTorch names the DECAY argument "alpha" (default 0.99) and the learning rate "lr".
w_mine = torch.randn(5, 3, requires_grad=True)
w_ref = w_mine.detach().clone().requires_grad_(True)          # identical start
opt_mine = MyRMSprop([w_mine], lr=1e-2, rho=0.99)
opt_ref = torch.optim.RMSprop([w_ref], lr=1e-2, alpha=0.99, eps=1e-8)  # alpha == our rho

X = torch.randn(20, 5)
target = torch.randn(20, 3)
for _ in range(6):
    opt_mine.zero_grad()
    (((X @ w_mine) - target) ** 2).mean().backward()
    opt_mine.step()
    opt_ref.zero_grad()
    (((X @ w_ref) - target) ** 2).mean().backward()
    opt_ref.step()

print("allclose vs torch.optim.RMSprop after 6 steps:",
      torch.allclose(w_mine, w_ref, atol=1e-7))             # expect True
print("max abs diff:", (w_mine - w_ref).abs().max().item()) # ~0

### Step 3 — Recompute the one-step worked example

Finally we reproduce the slide's hand-computed example so the code matches the lesson text: from `theta0=0`, `s0=0`, with gradient `g=0.1`, `rho=0.9`, `lr=0.01`, one step gives `s1 = 0.001`, `sqrt(s1) = 0.0316`, and a step of `0.0316`. Notice the gradient's magnitude cancels — that scale-normalization is the same intuition Adam later builds on.

In [ ]:
# Recompute the one-step worked example:
#   theta0=0, s0=0, g=0.1, t=1, rho=0.9, lr=0.01
th = torch.zeros(1, requires_grad=False)
o = MyRMSprop([th], lr=0.01, rho=0.9)     # rho=0.9 to match the slide's MeanSquare
th.grad = torch.tensor([0.1])
o.step()
print("worked example: s1=0.001, sqrt(s1)=0.0316228, step=0.0316228")
print("MyRMSprop new theta:", round(th.item(), 7))          # -0.0316228

## Visualize the data & results

_On a long run (120 steps) of the same convex least-squares problem from the same start, AdaGrad divides by sqrt(sum of all past squared gradients) and RMSProp divides by sqrt(a decaying average). Whose per-step movement decays to zero and stalls, and whose stays a healthy size and keeps lowering the loss?_

### Step 1 — Build a convex least-squares problem

To contrast AdaGrad and RMSProp we need a problem long enough for AdaGrad's growing denominator to bite. We generate a convex least-squares regression (`y = X w_true + noise`) and a helper that returns both the loss and its gradient at any weight vector.

In [ ]:
import numpy as np
rng = np.random.default_rng(0)

# convex least-squares; long horizon so AdaGrad's growing denominator bites
N, D = 200, 8
X = rng.normal(0, 1, (N, D))
w_true = rng.normal(0, 3, D)
y = X @ w_true + rng.normal(0, 0.01, N)

def loss_grad(w):
    r = X @ w - y
    return 0.5 * np.mean(r ** 2), (X.T @ r) / N

### Step 2 — Code the two optimizers side by side

The only difference between the two is how they accumulate squared gradients. AdaGrad uses a running **sum** `G += g*g`, which grows without bound. RMSProp uses a **decaying average** `s = rho*s + (1-rho)*g*g`, which stays bounded near the recent mean. Both then divide the gradient by the square root of that accumulator. We track the per-step update length so we can later see whose steps shrink to nothing.

In [ ]:
def run_adagrad(lr, steps=120, eps=1e-8):
    w = np.zeros(D)
    G = np.zeros(D)
    loss = []
    step = []
    for _ in range(steps):
        L, g = loss_grad(w)
        loss.append(L)
        G += g * g                            # SUM of past squared gradients (grows forever)
        upd = lr * g / (np.sqrt(G) + eps)
        step.append(np.linalg.norm(upd))
        w -= upd
    return loss, step

def run_rmsprop(lr, steps=120, rho=0.99, eps=1e-8):
    w = np.zeros(D)
    s = np.zeros(D)
    loss = []
    step = []
    for _ in range(steps):
        L, g = loss_grad(w)
        loss.append(L)
        s = rho * s + (1 - rho) * g * g       # DECAYING average (bounded) — the RMSProp change
        upd = lr * g / (np.sqrt(s) + eps)
        step.append(np.linalg.norm(upd))
        w -= upd
    return loss, step

### Step 3 — Run both and compare

Run each optimizer for 120 steps from the same zero start and compare the final loss and the last step length. AdaGrad's step has decayed to near zero and it stalls at a high loss; RMSProp's step is still a healthy size, so it keeps descending to a much lower loss. That single change — sum to decaying average — is what RMSProp contributed.

In [ ]:
ada_loss, ada_step = run_adagrad(lr=0.05)
rms_loss, rms_step = run_rmsprop(lr=0.01, rho=0.99)
print("AdaGrad final loss:", round(ada_loss[-1], 3),   # ~17.459
      "| last step len:", round(ada_step[-1], 5))       # ~0.00949
print("RMSProp final loss:", round(rms_loss[-1], 3),    # ~7.196
      "| last step len:", round(rms_step[-1], 5))        # ~0.01979

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** Do one RMSProp step ($\rho=0.9$, $\alpha=0.01$, $\epsilon=10^{-8}$) from $\theta_0=0$, $s_0=0$, but with a much larger gradient $g_1=10$ at $t=1$. How big is the step, and what does that tell you?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- MeanSquare: $s_1=0.9\cdot 0 + 0.1\cdot 10^2 = 0.1\cdot 100 = 10$. — _Average of the one squared gradient._
- RMS denominator: $\sqrt{s_1}+\epsilon=\sqrt{10}+\epsilon\approx 3.16228$. — _The recent gradient magnitude._
- Step: $0.01\cdot 10/3.16228\approx 0.0316228$. New $\theta_1\approx -0.0316228$. — _Gradient over its own size._

**Answer:** The step is again about $0.0316$ — identical to the $g_1=0.1$ case in the worked example. On the very first step $s_1=(1-\rho)g_1^2$, so $\sqrt{s_1}=\sqrt{1-\rho}\,|g_1|$ and the gradient's magnitude cancels in $g_1/\sqrt{s_1}$. The first step is about $\alpha/\sqrt{1-\rho}$ regardless of whether the gradient is 0.1 or 10. This scale-normalization is the same intuition behind Adam.

</details>

**Problem 2.** Why does RMSProp keep making progress on a long run where AdaGrad stalls?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- AdaGrad's denominator $\sqrt{G_t}$ with $G_t=\sum_{i\le t}g_i^2$ grows like $\sqrt{t}$. — _It sums every past squared gradient forever._
- So AdaGrad's step factor $\alpha/\sqrt{G_t}$ shrinks toward 0 — the step vanishes. — _A growing denominator kills the step._
- RMSProp's $s_t$ is a decaying average that settles near the recent mean squared gradient $c$, a bounded value. — _Old terms decay away, so the average does not climb._
- So RMSProp's step factor $\alpha/(\sqrt{s_t}+\epsilon)\approx \alpha/\sqrt{c}$ stays finite. — _A bounded denominator keeps a healthy step._

**Answer:** AdaGrad's accumulator $G_t$ only grows, so its effective step decays like $1/\sqrt{t}$ and freezes before reaching the minimum. RMSProp replaces the growing sum with a decaying moving average $s_t$ that stays bounded, so its step stays a healthy size and keeps lowering the loss. In our CODEVIZ run AdaGrad stalls near loss ~17.5 while RMSProp continues down to ~7.3 in the same 120 steps.

</details>

**Problem 3.** Ablation: in the CODE, replace RMSProp's moving average $s_t=\rho s_{t-1}+(1-\rho)g_t^2$ with AdaGrad's running SUM $G_t=G_{t-1}+g_t^2$ (keep everything else identical) and rerun the allclose against torch.optim.RMSprop. What happens, and why?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Change the state update to $G_t=G_{t-1}+g_t^2$ (no $\rho$, no $1-\rho$). — _That is AdaGrad's accumulator._
- Keep the step $\theta_t=\theta_{t-1}-\alpha g_t/(\sqrt{G_t}+\epsilon)$ and run the same 6 steps. — _Only the accumulator changed._
- Call $\texttt{torch.allclose}$ against $\texttt{torch.optim.RMSprop}$ and inspect the step sizes over time. — _PyTorch keeps the moving average._

**Answer:** The allclose now returns False. With the running sum, the denominator grows every step instead of settling, so after a few steps the step sizes diverge from PyTorch's RMSProp (they shrink faster). This is exactly the AdaGrad-vs-RMSProp difference: the single change from "sum" to "decaying average" is what RMSProp added, and it is what stops the step from vanishing. The mismatch grows with $t$ as the AdaGrad denominator keeps climbing.

</details>